In [ ]:
import streamlit as st
import sqlite3
from datetime import datetime


# DATABASE SETUP AND FUNCTIONS


def create_db():
    conn = sqlite3.connect('DatabaseName')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS ev_battery (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            vehicle_owner TEXT,
            ev_model TEXT,
            petrol_model TEXT,
            battery_cost REAL,
            petrol_vehicle_cost REAL,
            purchase_date TEXT,
            warranty_end TEXT
        )
    ''')
    conn.commit()
    conn.close()

def insert_data(owner, ev_model, petrol_model, battery_cost, petrol_cost, purchase_date, warranty_end):
    conn = sqlite3.connect('ev_data.db')
    c = conn.cursor()
    c.execute('''
        INSERT INTO ev_battery (vehicle_owner, ev_model, petrol_model, battery_cost, petrol_vehicle_cost, purchase_date, warranty_end)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (owner, ev_model, petrol_model, battery_cost, petrol_cost, purchase_date, warranty_end))
    conn.commit()
    conn.close()

def fetch_data():
    conn = sqlite3.connect('ev_data.db')
    c = conn.cursor()
    c.execute('SELECT * FROM ev_battery')
    data = c.fetchall()
    conn.close()
    return data


# STREAMLIT FRONTEND


# Initialize DB
create_db()

st.title("EV vs Petrol Vehicle Cost & Battery Warranty Analysis")

# ---- Input Form ----
st.subheader("Enter Vehicle Information")

owner = st.text_input("Owner Name")
ev_model = st.text_input("EV Car Model")
petrol_model = st.text_input("Petrol Car Model")

battery_cost = st.number_input("EV Battery Cost ($)", min_value=0.0, format="%.2f")
petrol_cost = st.number_input("Petrol Vehicle Cost ($)", min_value=0.0, format="%.2f")

purchase_date = st.date_input("EV Battery Purchase Date")
warranty_end = st.date_input("Warranty Expiry Date")

if st.button("Submit"):
    insert_data(owner, ev_model, petrol_model, battery_cost, petrol_cost,
                purchase_date.isoformat(), warranty_end.isoformat())
    st.success("Data saved successfully!")

# ---- Display Stored Data ----
st.subheader("Stored Records")

data = fetch_data()
if data:
    for row in data:
        st.markdown(f"""
        *Owner:* {row[1]}  
        *EV Model:* {row[2]}  
        *Petrol Model:* {row[3]}  
        *Battery Cost:* ${row[4]}  
        *Petrol Vehicle Cost:* ${row[5]}  
        *Purchase Date:* {row[6]}  
        *Warranty Ends:* {row[7]}  
        """)
else:
    st.info("No records found.")

# ---- Cost Comparison ----
st.subheader("Live Cost Comparison")

if ev_model and petrol_model and battery_cost and petrol_cost:
    st.markdown(f"Comparing *{ev_model}* vs *{petrol_model}*:")
    if battery_cost < petrol_cost:
        st.success(f"{ev_model}'s battery is cheaper than {petrol_model}.")
    elif battery_cost > petrol_cost:
        st.warning(f"{ev_model}'s battery is more expensive than {petrol_model}.")
    else:
        st.info("Both have the same cost.")

# ---- Optional Warranty Reminder ----
st.subheader("Warranty Status")

today = datetime.now().date()
if warranty_end < today:
    st.error("Warranty has expired!")
elif (warranty_end - today).days <= 30:
    st.warning("Warranty is about to expire soon.")
else:
    st.success("Warranty is still valid.")